# 02 Baselines — Stage A
Trains logistic regression and XGBoost (values-only vs values+timing) on the source environment and evaluates in-environment + cross-site. Saves results as JSONL.

In [ ]:
import os, json
from pathlib import Path
from vist_nrf.utils.seed import set_seed
from vist_nrf.models import build_model
from vist_nrf.eval.metrics import evaluate

set_seed(0)
results_dir = Path(os.environ.get('VIST_NRF_RESULTS', '/content/drive/MyDrive/VIST-NRF/results'))
results_dir.mkdir(parents=True, exist_ok=True)

# Synthetic smoke test (replace with real PhysioNet data path after download)
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(0, 1, (200, 12))
y = (X[:, 0] + 0.5 * X[:, 3] + rng.normal(0, 0.5, 200) > 0).astype(int)
envs = [0] * 100 + [1] * 100

for name in ['logistic_regression', 'xgboost_values', 'xgboost_timing']:
    try:
        m = build_model(name, seed=0)
        m.fit(X[:100], y[:100], envs=envs[:100])
        p = m.predict_proba(X[100:])[:, 1]
        mres = evaluate(y[100:], p)
        print(name, '| AUROC=%.3f AUPRC=%.3f Brier=%.3f' % (mres.auroc, mres.auprc, mres.brier))
    except Exception as e:
        print(name, 'SKIPPED:', e)